In [ ]:
import os  # reads settings from the environment

import pandas as pd  # pandas, always nicknamed "pd"
from dotenv import load_dotenv  # loads .env, same as loader.py
from sqlalchemy import URL, create_engine  # the connection pandas prefers

pd.set_option("display.max_columns", None)  # show all columns, never "..."

load_dotenv()

url = URL.create(
    "postgresql+psycopg",                   # Postgres, through psycopg
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),         # must be a number here, not text
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)                 # a reusable door into the database

In [ ]:
query = "SELECT * FROM pd_features_build WHERE sample_set = 'train'"
df = pd.read_sql(query, engine)   # run the query, keep the result as a table
df.shape                          # (rows, columns)

In [ ]:
print(df.head(10).T.to_string())

In [ ]:
df.info()

In [ ]:
print(df.describe().T.to_string())

In [ ]:
for col in ["amortization_type", "prepayment_penalty_indicator",
            "interest_only_indicator", "property_type", "channel"]:
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
print(df["msa"].isna().mean())
print(df["msa"].nunique())
print(df["msa"].value_counts().head(10))

In [ ]:
print(df["postal_code"].nunique())
print(df["property_state"].nunique())

In [ ]:
print("--- check 3: LTV over 100 ---")
high_ltv = df[df["original_ltv"] > 100]
print(len(high_ltv))
print(high_ltv["harp_indicator"].value_counts())

print("--- check 4: rate spread range, and the odd 276/288 terms ---")
print(df["rate_spread"].describe())
print(df[df["loan_term"].isin([276, 288])]["harp_indicator"].value_counts())

print("--- check 5: credit scores at the very bottom ---")
print((df["credit_score"] == 300).sum())
print((df["credit_score"] < 500).sum())

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
print(pd.DataFrame({"missing": missing, "pct": (missing / len(df) * 100).round(3)}))

In [ ]:
fill_numbers = ["credit_score", "number_of_borrowers", "number_of_units",
                "original_cltv", "original_ltv", "mi_percentage"]
print(df[fill_numbers].median())
print(df["first_time_homebuyer"].mode())
print(df["property_type"].mode())

print(df.groupby(df["original_dti_ratio"].isna())["target"].agg(["count", "mean"]))
